<a href="https://colab.research.google.com/github/Annpeng1005/MLH-Crypto-Tracker/blob/main/Sprint_04_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 4: RAG Pipeline (Retrieval-Augmented Generation)

Welcome to the final boss of your MLH project!

Currently, our AI is "blind." It knows the price dropped, but it doesn't know *why*. Today, we are going to give it eyes by fetching live news articles and feeding them directly into our prompt.

This technique is called **RAG**, and it is how modern PMs build intelligent features that don't hallucinate.

---

## The Problem (The "Why")
When a coin crashes, we need to automatically read the news to find out why. We will use an API from CryptoCompare that returns news articles based on a coin's symbol.

Because cloud servers share IP addresses, CryptoCompare will block anonymous requests from Colab. You need an API key to prove you are an authenticated developer.

---

## Your Acceptance Criteria

1. **Get the Key:** Go to [CryptoCompare](https://www.cryptocompare.com/cryptopian/api-keys), create a free account, and get an API key. Save it in your Colab Secrets tab exactly as `CRYPTO_API_KEY`.
2. **Defensive Setup:** Write an `if not crypto_key:` statement right after you load it. If the key is empty, use `raise ValueError("Key is missing!")` to crash the script safely rather than pinging the API with a blank key.
3. **Construct the Header:** In production environments, it is a security risk to put API keys directly in the URL. Instead, we use HTTP Headers.
   * Create a dictionary: `headers = {"Authorization": f"Apikey {crypto_key}"}` *(Note the capital 'A' in Authorization!)*
4. **Trigger the News:** Inside your existing `if daily_change <= -5.0:` block, make your request using a 2-second pause (`time.sleep(2)`) to avoid rate limits.
   * *URL format:* `f"https://min-api.cryptocompare.com/data/v2/news/?lang=EN&categories={coin_symbol}"`
   * *The Call:* `requests.get(news_url, headers=headers).json()`
5. **Upgrade the Prompt:** Rewrite your Gemini prompt. Pass in the coin name, the price drop, AND the 3 headlines. Ask Gemini to write a 2-sentence Slack alert explaining the drop using the provided news context.
6. **Export:** Append this new alert to your `market_alerts.txt` file.

---

## Final Version Control
Once your AI is accurately summarizing the news, commit and push your final notebook to your GitHub repository. You now have a complete, AI-powered data engineering portfolio piece ready for your MLH application!

In [1]:
import requests
import time
from google import genai
from google.colab import userdata
import csv
from os import times_result

In [2]:
gemini_api_key = userdata.get('gemini_api_key')
crypto_api_key = userdata.get('crypto_api_key')

headers = {"Authorization": f"Apikey {crypto_api_key}"}

In [3]:
client = genai.Client(api_key= gemini_api_key)

In [4]:
def make_ai_alert(coin_name, change, price, titles):

  headline_text = ""
  for title in titles:
    headline_text = headline_text + title + "\n"

  prompt = f"""
  Act as financial analyst,

  coin:{coin_name}
  24-hour change: {change}%
  Current price: ${price}

  Recent news headlines:
  {headline_text}

  Write a professional, urgent 2-sentence Slack alert explaining the price drop using only the news conext above.
  """

  response = client.models.generate_content(
      model="gemini-2.5-flash", contents = prompt)

  ai_alert = response.text
  return ai_alert

In [5]:
target_coins = ["bitcoin", "ethereum", "solana", "ripple", "dogecoin"]

def get_roster_data():
  url_roster = "https://api.coingecko.com/api/v3/coins/list"

  roster_response = requests.get(url_roster)
  roster_data = roster_response.json()

  return roster_data
roster_data = get_roster_data()

In [6]:
def get_url_live():

  url_live = "https://api.coingecko.com/api/v3/simple/price"

  price_params= {'ids': 'bitcoin,ethereum,solana,ripple,dogecoin',
                'vs_currencies': 'usd',
                  'include_24hr_change':True
                }

  price_response = requests.get(url_live, params=price_params)
  price_data = price_response.json()

  return price_data

price_data = get_url_live()

In [7]:
def filter_coins_roster(roster_data, target_coins):
  filter_coins=[]
  for el in roster_data:
    if el['id'] in target_coins:
      filter_coins.append(el)
  return filter_coins

filter_coins = filter_coins_roster(roster_data, target_coins)

In [13]:

def get_news(base_news_url, coin_symbol):
  time.sleep(2)
  news_url = f"{base_news_url}/?lang=EN&categories={coin_symbol}"
  news_response = requests.get(news_url, headers=headers)
  news_data = news_response.json()

  return news_data

def get_top_headlines(news_data):
# at least 3 news or the minimum news that's accessable
  top_3_news=[]
  number_of_news_items = min(3, len(news_data["Data"]))
  for i in range(number_of_news_items):
    top_3_news.append(news_data["Data"][i]["title"])

  return top_3_news

def handle_alert(base_news_url, coin, change, price):
  coin_symbol = coin["symbol"].upper()
  news_data = get_news(base_news_url, coin_symbol)

  top_3_news = get_top_headlines(news_data)
  ai_alert = make_ai_alert(coin["name"], change, price, top_3_news)

  return ai_alert

def analyze_market_data():
  alerts = []
  base_news_url = "https://min-api.cryptocompare.com/data/v2/news"

  # threshold is flexible for testing
  threshold = 0

  for coin in filter_coins:
      for price_id in price_data:
          if coin['id'] == price_id:
              change = price_data[price_id]['usd_24h_change']
              price = price_data[price_id]['usd']

              if change <= threshold:
                  ai_alert = handle_alert(base_news_url, coin, change, price)
                  print(ai_alert)
                  alerts.append(ai_alert)

  return alerts

  alerts = analyze_market_data()

*Urgent: Bitcoin's 24-hour price drop reflects immediate pressure from the looming $6 billion options expiry risk. This is exacerbated by negative sentiment from headlines regarding the crypto industry's health and significant BTC revenue declines reported by companies like Iren.*


KeyboardInterrupt: 

In [11]:
with open('market_alerts.txt', "w") as file:
  if len(alerts) == 0:
    file.write('Market is stable today.')
  else:
    for alert in alerts:
      file.write(alert)
      if alert not in alerts[-1]:
        file.write("\n"+"-"+"\n")

print("market_alerts.txt is imported successfully.")

market_alerts.txt is imported successfully.
